# Serve + Test the E4B adapter on Colab T4 (HF nf4 — full fidelity)

Serves your trained E4B LoRA at the SAME nf4 4-bit precision it was trained on (no GGUF
quant degradation), on a free Colab **T4 (16GB)** which fits E4B 4-bit with room. This is
the truest read of model quality — q4_0 GGUF would confound it (and we've seen q4_0
fabricate numbers).

## Run order
1–6 setup → **6c** (shared model helpers) → **7** probe → **7a** plan-mode probe →
**7b** generalization → **7c** confusion matrix + precision → **8** web UI (browser chat).

## OOM is designed out
A T4 fits exactly **one** E4B. Every eval/probe cell gets the model through `get_model()`
(Cell 6c): it loads the model **once**, **reuses** it on every later cell, and frees any
subprocess service first — so re-running a cell, or running cells in any order, never makes
a **second** model resident. The web-UI cell (8) does the reverse: it `free_model()`s the
in-kernel model before starting its subprocess service. That mutual-exclusion is why the
"load HFModel twice → OOM" you hit before cannot recur.

**Before you run:** Runtime → Change runtime type → **T4 GPU**. Add Colab Secrets:
`HF_TOKEN` (HF read) and `NGROK_TOKEN` (free from ngrok.com → authtoken).

In [ ]:
# Cell 1 - config
import os
BASE_REPO = "unsloth/gemma-4-E4B-it"   # the base your adapter was trained on
REPO_URL  = "https://github.com/RyanDev1st/llm-and-engine.git"
BRANCH    = "feat/chess-coach-sft"

if os.path.exists("A:/Download/llm_tool_calling_research_workspace"):
    WORKDIR = "A:/Download/llm_tool_calling_research_workspace"
    ADAPTER_DIR = "A:/Download/gemma4_chess_e4b_kaggle (1)"
else:
    WORKDIR = "/content/llm-and-engine"
    ADAPTER_DIR = "/content/adapter"

BASE_DIR  = f"{WORKDIR}/src/llm/models/gemma4_e4b"
print("base:", BASE_REPO, "| adapter:", ADAPTER_DIR)

In [ ]:
# Cell 2 - GPU check (want a T4 16GB)
import subprocess, torch
print(subprocess.run(["nvidia-smi","--query-gpu=name,memory.total,memory.free","--format=csv"],
                     capture_output=True, text=True).stdout)
assert torch.cuda.is_available(), "No GPU — Runtime > Change runtime type > T4 GPU"
print("device:", torch.cuda.get_device_name(0))

In [ ]:
# Cell 3 - clone repo
import os, subprocess
env = {**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"}
if not os.path.exists(WORKDIR):
    subprocess.run(["git","clone","--depth","1","--branch",BRANCH,REPO_URL,WORKDIR], check=True, env=env)
else:
    subprocess.run(["git","-C",WORKDIR,"pull","--ff-only"], check=True, env=env)
subprocess.run(["git","-C",WORKDIR,"log","-1","--oneline"])
os.environ["PYTHONPATH"] = f"{WORKDIR}/src/llm"
print("PYTHONPATH=", os.environ["PYTHONPATH"])

In [ ]:
# Cell 4 - deps (serve path + ngrok + stockfish for full chess tools)
import subprocess, sys, shutil
def pip(*a): subprocess.run([sys.executable,"-m","pip","install","-q",*a], check=True)
pip("-U","transformers","accelerate","peft","bitsandbytes","sentencepiece","protobuf","python-chess","pyngrok")
if shutil.which("apt-get"):
    subprocess.run(["apt-get","-qq","install","-y","stockfish"])  # chess tools (optional)
import transformers, peft, bitsandbytes
print("transformers",transformers.__version__,"| peft",peft.__version__,"| bnb",bitsandbytes.__version__)

In [ ]:
# Cell 5 - HF login + download the E4B base (~9GB; once)
import os
from huggingface_hub import login, snapshot_download
try:
    from google.colab import userdata
    login(userdata.get("HF_TOKEN"))
except Exception:
    login(os.environ["HF_TOKEN"])
snapshot_download(repo_id=BASE_REPO, local_dir=BASE_DIR,
                  allow_patterns=["*.json","*.safetensors","*.model","*.txt","tokenizer*"])
print("base at", BASE_DIR, "->", sorted(os.listdir(BASE_DIR))[:8])

In [ ]:
# Cell 6 - download the adapter from Hugging Face (FORMAT-FIXED run = ckpt-v4)
# v4 run: all-linear + FORMAT_WEIGHT=8.0 on the control tags + the serve-side </skill>
# stop fix. best/ = lowest-val checkpoint. val is NOT the judge here — Cell 7 (train-row
# free-gen probe) and Cell 7c (routing confusion matrix + precision) are.
#   TAG='best'       -> lowest-val adapter (PROBE/SERVE THIS)
#   TAG='checkpoint' -> latest step (only for resume)
import os
from huggingface_hub import snapshot_download

ADAPTER_REPO = "RyanDev1st/gemma4-chesscoach-ckpt-v4"   # current run (v3/v2/v1 superseded)
TAG = "best"                                            # lowest-val checkpoint

if not os.path.exists("A:/Download/llm_tool_calling_research_workspace"):
    print(f"Downloading adapter from HF: {ADAPTER_REPO} :: {TAG}/")
    snapshot_download(repo_id=ADAPTER_REPO, local_dir="/content/hf_adapter_v4",
                      allow_patterns=[f"{TAG}/*"])
    ADAPTER_DIR = f"/content/hf_adapter_v4/{TAG}"

need = ("adapter_model.safetensors", "adapter_config.json")
ok = all(os.path.exists(f"{ADAPTER_DIR}/{f}") for f in need)
if os.path.exists(ADAPTER_DIR):
    print("adapter at", ADAPTER_DIR, "->", os.listdir(ADAPTER_DIR))
    # sanity: all-linear adapter MUST have MLP modules (gate/up/down), not just attn
    import json as _j
    cfg = _j.loads(open(f"{ADAPTER_DIR}/adapter_config.json").read())
    print("target_modules:", cfg.get("target_modules"), "| r:", cfg.get("r"),
          "| alpha:", cfg.get("lora_alpha"), "| dropout:", cfg.get("lora_dropout"))
else:
    print("adapter at", ADAPTER_DIR, "-> NOT FOUND")
assert ok, f"adapter not found at {ADAPTER_DIR}"

In [ ]:
# Cell 6b - (alt) upload the adapter instead of Drive. Zip best/ locally, upload here.
# from google.colab import files; import zipfile, io, os
# up = files.upload()                       # pick best.zip
# name = next(iter(up))
# ADAPTER_DIR = "/content/best"
# with zipfile.ZipFile(io.BytesIO(up[name])) as z: z.extractall("/content")
# print(os.listdir(ADAPTER_DIR))

In [ ]:
# Cell 6c - SHARED in-kernel model + OOM guard (run this ONCE; every eval/probe cell reuses it).
# The T4 fits exactly ONE E4B. To make re-running ANY cell (in any order) safe, model access goes
# through get_model(), which:
#   1. frees any SUBPROCESS model (the web-UI service) so it can't co-reside,
#   2. reuses the already-loaded in-kernel MODEL if it is alive,
#   3. else loads it ONCE.
# The web-UI cell does the reverse (free_model() before it starts the subprocess service). So no
# two E4Bs are ever resident -> the "load HFModel twice" OOM cannot happen.
import os, sys, gc, subprocess
import torch
sys.path.insert(0, f'{WORKDIR}/src/llm')
os.environ['CHESS_HF_BASE'] = BASE_DIR
from backend.inference import build_system_prompt          # served default catalog
from llm_training.system_prompt import build_system        # per-row prompt (same builder training used)


def _free_subprocess_models():
    # the web-UI model service / app server hold the GPU in a subprocess; kill them so an
    # in-kernel load can't make a SECOND E4B resident.
    subprocess.run(['pkill', '-f', 'backend.model_server'])
    subprocess.run(['pkill', '-f', 'backend.server'])


def free_model():
    """Drop the in-kernel MODEL and free its VRAM (the web-UI cell calls this before it starts
    the subprocess service, so the service's model isn't a SECOND resident E4B)."""
    if globals().get('MODEL') is not None:
        del globals()['MODEL']
    gc.collect(); torch.cuda.empty_cache()
    print('in-kernel MODEL freed (VRAM released)')


def get_model():
    """The ONE in-kernel HFModel — loaded once, reused everywhere. Frees any subprocess model
    first, and reuses a live MODEL, so a second run / a different cell never makes two E4Bs
    resident (the OOM). Returns the shared MODEL."""
    m = globals().get('MODEL')
    if m is not None:
        try:
            m.generate([{'role': 'user', 'content': 'hi'}], 2, [])      # is it still alive?
            return m
        except Exception:
            globals()['MODEL'] = None                                    # dead -> reload below
    _free_subprocess_models()
    gc.collect(); torch.cuda.empty_cache()
    from backend.model_hf import HFModel
    print('loading E4B base + v4 adapter in-kernel (ONCE, ~1-2 min)...', flush=True)
    globals()['MODEL'] = HFModel(base=BASE_DIR, adapter=ADAPTER_DIR, temperature=0.0)
    print('MODEL ready (in-kernel, reused by every cell below)')
    return globals()['MODEL']


def gen(sysp, user, mx=160, stop=('</tool>', '</skill>', '</plan>')):
    """One generation against the shared in-kernel MODEL — drop-in for the old HTTP _gen()."""
    return get_model().generate(
        [{'role': 'system', 'content': sysp}, {'role': 'user', 'content': user}], mx, list(stop))


print('shared model helpers ready: get_model() | gen(sysp,user) | free_model()')

In [ ]:
# Cell 7 - PROBE (DECISIVE). Two tests against the v4 adapter, IN-KERNEL via the shared MODEL
# (get_model() — loads once, reused by every cell; no subprocess, no OOM).
#  (A) TRAIN-ROW REPRODUCTION — feed the EXACT training-row system prompt (per-row skills index
#      + tool manifest) and the row's user turn, compare to the gold assistant turn. If it can't
#      emit the format on a row it TRAINED on, format-emission is broken. Auto-classified:
#        OK     = emits <skill>/<tool>, NO foreign tags
#        SOUP   = foreign tags (<action>/<thought>/<unused..>) = format collapse
#        NO-EMIT= no <skill>/<tool> at all
#  (B) GENERIC SERVE — real-world prompts via the default served catalog (what a user sees).
import json, re, gzip

get_model()   # load the shared model ONCE (reused by 7a/7b/7c); frees any subprocess service first

# ---------- (A) TRAIN-ROW REPRODUCTION ----------
CONTRACT={'think','/think','goal','/goal','plan','/plan','skill','/skill','tool','/tool'}
def classify(out):
    tags=re.findall(r'</?([a-zA-Z_][\w]*)', out)
    foreign=sorted({t for t in tags if t not in CONTRACT})
    emits=('<skill>' in out) or ('<tool>' in out)
    return ('OK' if emits and not foreign else 'SOUP' if foreign else 'NO-EMIT'), foreign

rows=[]
p=f"{WORKDIR}/data/sft/v1_2_train.jsonl"
if not os.path.exists(p): p+='.gz'
op=gzip.open if p.endswith('.gz') else open
with op(p,'rt',encoding='utf-8') as f:
    for i,line in enumerate(f):
        if i>=800: break
        line=line.strip()
        if line: rows.append(json.loads(line))
def pick(mode,n): return [r for r in rows if r.get('reasoning_mode')==mode][:n]
samples=pick('fast',2)+pick('think',2)+pick('auto',2)
print('\n========== (A) TRAIN-ROW REPRODUCTION (v4 adapter) ==========')
tally={'OK':0,'SOUP':0,'NO-EMIT':0}
for r in samples:
    sysp=build_system(r['skills_index'],r['tool_manifest'],r.get('plugin_context',{}),
                      reasoning_mode=r.get('reasoning_mode',''))
    user=next(m['content'] for m in r['messages'] if m['role']=='user')
    gold=next(m['content'] for m in r['messages'] if m['role']=='assistant')
    out=gen(sysp,user); v,foreign=classify(out); tally[v]+=1
    print('='*64); print(f"[{r.get('reasoning_mode')}] slice={r.get('slice')} -> {v}   foreign={foreign}")
    print('USER :',user[:90]); print('GOLD :',gold[:150]); print('MODEL:',out[:300])
print('\nTRAIN-ROW TALLY:',tally,
      '  => verdict:', 'WIN' if tally['OK']>=4 else
                       'SOUP=format collapse; data problem' if tally['SOUP']>=3 else 'MIXED')

# ---------- (B) GENERIC SERVE (real-world) ----------
print('\n========== (B) GENERIC SERVE ==========')
def step(prompt, mode):
    out=gen(build_system_prompt(reasoning_mode=mode), prompt, mx=220)
    print(f'### [{mode}] {prompt}'); print(out); print('-'*70)
step('I scored 70, 10, and 8 - is my average above 85?', 'think')
step('What is the best move here?', 'auto')
step('hey, what can you do?', 'fast')

In [ ]:
# Cell 7a - PLAN-MODE behavioral probe: does the v4 adapter actually emit <goal>/<plan> on the
# plan signal? The earlier probes only sampled fast/think/auto, so plan was NEVER exercised — but
# the corpus DOES train it: V1_S_compound_plan + V1_T_audited_plan, ~2.2k train rows (3%), every
# one a concrete <goal> + <plan> checklist. Uses the shared in-kernel MODEL (gen()). Expect a
# <goal> then a <plan> of '- [ ] action (binding)' boxes (the serve plan-gate + UI render exactly
# these), then tool steps.
PLAN_PROMPTS = [
    'two things: first tell me who is winning, then lay out a plan to improve my position',
    'check my average of 86, 82, 76 is over 80, and also whether $38.20 split 3 ways is under $15 each',
    'help me with two things: review my last move, then suggest the best continuation',
]
print('\n========== PLAN MODE (does it emit <goal>/<plan>?) ==========')
ok = 0
for p in PLAN_PROMPTS:
    out = gen(build_system_prompt(reasoning_mode='plan'), p, mx=220, stop=('</plan>', '</tool>', '</skill>'))
    has_plan = '<plan>' in out
    ok += has_plan
    print(f"### [plan] emits<plan>={has_plan} <goal>={'<goal>' in out} | {p}")
    print(out); print('-' * 70)
print(f'PLAN tally: {ok}/{len(PLAN_PROMPTS)} emitted a <plan> checklist',
      '=> plan mode WORKS' if ok >= 2 else
      '=> plan rarely emitted — inspect (corpus trains it; may need more plan rows/steps or it is mode-mixing)')

In [ ]:
# Cell 7c - CONFUSION MATRIX + PRECISION (LIGHT + interrupt-safe, for the slides). IN-KERNEL via
# the shared MODEL (get_model() — reuses the loaded model; never a 2nd E4B, so no OOM). Routing
# scored as a 3-class verb decision (skill/tool/none) on the model's FIRST action vs gold, over a
# STRATIFIED sample. Scored in FAST mode (routing is mode-independent; fast skips the goal/think
# preamble so each row acts in a few tokens — ~5x faster than the ~20s/row full-reasoning path).
# TIME_BUDGET stops early and still writes the matrix from completed rows. Raise PER_SLICE later.
import sys
from datetime import date
from pathlib import Path
sys.path.insert(0, f'{WORKDIR}/src/llm')
from llm_training.eval_confusion import evaluate, _metrics, _png, _sample, CLASSES
from llm_dataset.v1.jsonl_io import read_rows

MODEL = get_model()        # reuse the shared in-kernel model (loads once); frees any subprocess model
PER_SLICE      = 8         # rows/slice (~250 rows). At ~3-4s/row in fast mode => ~12-18 min.
MAX_NEW_TOKENS = 24        # fast mode acts immediately; a 'none' row stops at this cap
TIME_BUDGET_S  = 15 * 60   # hard stop; writes the matrix from whatever finished (survives a kill)

rows = _sample(list(read_rows(f'{WORKDIR}/data/sft/v1_2_val.jsonl')), PER_SLICE)
print(f'evaluating {len(rows)} rows (per_slice={PER_SLICE}, fast mode, budget={TIME_BUDGET_S//60}min) ...')
cm, (nh, nt), (sc, st) = evaluate(MODEL, rows, max_new_tokens=MAX_NEW_TOKENS,
                                  time_budget_s=TIME_BUDGET_S, force_fast=True)
met = _metrics(cm)
tot = sum(cm[g][p] for g in CLASSES for p in CLASSES); acc = sum(cm[c][c] for c in CLASSES) / max(tot, 1)
macro = sum(met[c]['precision'] for c in CLASSES) / 3
print(f"\nscored {tot} rows | verb-acc {acc:.1%} | macro-precision {macro:.1%} | "
      f"exact-name {nh}/{nt}={nh/max(nt,1):.1%}\n")
print("gold\\pred", CLASSES)
for g in CLASSES: print(f"{g:>5}", [cm[g][p] for p in CLASSES])
for c in CLASSES: print(c, {k: round(v, 3) for k, v in met[c].items()})

out_png = Path(f'{WORKDIR}/docs/findings/{date.today():%Y-%m-%d}-routing-confusion.png')
out_png.parent.mkdir(parents=True, exist_ok=True)
_png(cm, out_png)
if out_png.exists():
    from IPython.display import Image, display
    display(Image(filename=str(out_png))); print('saved:', out_png)

In [ ]:
# Cell 7c - CONFUSION MATRIX + PRECISION from the v4 adapter (for the slides). IN-KERNEL via the
# shared MODEL (get_model() — reuses the already-loaded model; never loads a second E4B, so it
# cannot OOM). Routing scored as a 3-class verb decision on the model's FIRST action vs the gold
# first action over the validation set: skill / tool / none. Prints the 3x3 matrix (rows=gold,
# cols=pred) + per-class precision/recall/F1 + exact-name accuracy, and saves the PNG.
import sys
from datetime import date
from pathlib import Path
sys.path.insert(0, f'{WORKDIR}/src/llm')
from llm_training.eval_confusion import evaluate, _metrics, _png, _sample, CLASSES
from llm_dataset.v1.jsonl_io import read_rows

MODEL = get_model()        # reuse the shared in-kernel model (loads once); frees any subprocess model
PER_SLICE = 40             # rows/slice (~10-15 min). Set None for the FULL val set (~60 min, best number).

rows = _sample(list(read_rows(f'{WORKDIR}/data/sft/v1_2_val.jsonl')), PER_SLICE)
cm, (nh, nt), (sc, st) = evaluate(MODEL, rows)
met = _metrics(cm)
tot = sum(cm[g][p] for g in CLASSES for p in CLASSES); acc = sum(cm[c][c] for c in CLASSES) / tot
macro = sum(met[c]['precision'] for c in CLASSES) / 3
print(f"\nverb-acc {acc:.1%} | macro-precision {macro:.1%} | exact-name {nh}/{nt}={nh/nt:.1%}\n")
print("gold\\pred", CLASSES)
for g in CLASSES: print(f"{g:>5}", [cm[g][p] for p in CLASSES])
for c in CLASSES: print(c, {k: round(v, 3) for k, v in met[c].items()})

out_png = Path(f'{WORKDIR}/docs/findings/{date.today():%Y-%m-%d}-routing-confusion.png')
out_png.parent.mkdir(parents=True, exist_ok=True)
_png(cm, out_png)
if out_png.exists():
    from IPython.display import Image, display
    display(Image(filename=str(out_png))); print('saved:', out_png)

In [ ]:
# Cell 8 - SERVE the web UI (browser chat via ngrok). The web app needs the model in a SUBPROCESS
# service, so this first FREES the in-kernel MODEL (else two E4Bs = OOM), then starts the service
# + the weightless app. Re-running only restarts the app (the model service is reused).
import os, sys, subprocess, time, json, urllib.request
SVC = 'http://127.0.0.1:7861'

free_model()                          # release the in-kernel eval model so the service can load (OOM guard)

def _health():
    try: return json.load(urllib.request.urlopen(SVC+'/health', timeout=5)).get('ok')
    except Exception: return False

def ensure_service():
    if _health():
        print('model service already UP - reusing'); return
    env={**os.environ,'PYTHONPATH':f'{WORKDIR}/src/llm','CHESS_HF_ADAPTER':ADAPTER_DIR,'CHESS_HF_BASE':BASE_DIR}
    log=open('/content/model_server.log','w')
    globals()['_MSVC']=subprocess.Popen([sys.executable,'-u','-m','backend.model_server',ADAPTER_DIR],
                                        cwd=f'{WORKDIR}/src/llm',env=env,stdout=log,stderr=subprocess.STDOUT)
    print('starting model service with', ADAPTER_DIR, '(loads ONCE, ~1-2 min)...')
    for _ in range(180):
        time.sleep(2)
        if globals()['_MSVC'].poll() is not None:
            print(open('/content/model_server.log').read()[-3000:]); raise RuntimeError('service crashed')
        if _health(): print('model service UP'); return
    print(open('/content/model_server.log').read()[-3000:]); raise RuntimeError('service did not come up')

ensure_service()                      # start the one model service (or reuse it)
env={**os.environ,'PYTHONPATH':f'{WORKDIR}/src/llm','CHESS_HF_ADAPTER':ADAPTER_DIR,'CHESS_HF_BASE':BASE_DIR,
     'CHESS_MODEL_SERVER':SVC,'CHESS_HOST':'127.0.0.1','CHESS_PORT':'7860'}
subprocess.run(['pkill','-f','backend.server']); time.sleep(2)   # restart app only (cheap, weightless)
globals()['_APP']=subprocess.Popen([sys.executable,'-u','-m','backend.server'],cwd=f'{WORKDIR}/src/llm',
                  env=env,stdout=open('/content/app_server.log','w'),stderr=subprocess.STDOUT)
time.sleep(5)
from pyngrok import ngrok
ngrok.kill()                          # drop old tunnels (free tier allows 1)
try:
    from google.colab import userdata; ngrok.set_auth_token(userdata.get('NGROK_TOKEN'))
except Exception:
    ngrok.set_auth_token(os.environ['NGROK_TOKEN'])
url = ngrok.connect(7860)
print('==============================================')
print('  OPEN THIS IN YOUR BROWSER TO CHAT-TEST:')
print('  ', url)
print('==============================================')
print('--- app server log ---')
print(open('/content/app_server.log').read()[-1500:])